# 01 · Bronze ingestion
Raw CSVs from the landing volume → Delta tables, untouched.
Everything lands as string; typing happens in silver.

In [0]:
from pyspark.sql import functions as F

from easymoney_pipeline.quality import check_row_count
from easymoney_pipeline.transforms import clean_col

CATALOG = "workspace"
BRONZE = f"{CATALOG}.easymoney_bronze"
LANDING = f"/Volumes/{CATALOG}/easymoney_bronze/landing"

TABLES = [
    "customer_commercial_activity",
    "customer_products",
    "customer_sociodemographics",
    "sales",
    "product_description",
]

In [0]:
# Idempotent: safe to rerun from scratch
for layer in ("bronze", "silver", "gold"):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.easymoney_{layer}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {BRONZE}.landing")

In [0]:
def ingest_to_bronze(table: str) -> int:
    df = (spark.read
          .option("header", True)
          .option("inferSchema", False)
          .csv(f"{LANDING}/{table}.csv")
          .withColumn("_source_file", F.col("_metadata.file_path")))
    df = (df.toDF(*[clean_col(c) for c in df.columns])
            .withColumn("_ingested_at", F.current_timestamp()))

    n_source = df.count()
    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", True)
       .saveAsTable(f"{BRONZE}.{table}"))

    spark.sql(f"""COMMENT ON TABLE {BRONZE}.{table} IS
        'Raw copy of {table}.csv. All columns as string. Lineage: _source_file, _ingested_at.'""")

    # Source vs table reconciliation
    return check_row_count(spark.table(f"{BRONZE}.{table}"), n_source, f"bronze.{table}")

In [0]:
for t in TABLES:
    print(f"{t:<30} {ingest_to_bronze(t):>12,} rows")